# NetWeaver SRE — GRPO Training in Colab

This notebook is **self-contained** for Google Colab. It will:

1. Clone the `netweaver-sre` repo and install dependencies
2. Boot the FastAPI environment server in a background thread
3. Run a real evaluation **before** training (random/baseline policy)
4. Run GRPO training with `Qwen/Qwen2.5-0.5B-Instruct` against the live env
5. Run a real evaluation **after** training
6. Plot the reward curve, before/after bars, and per-difficulty breakdown

**Open in Colab → Runtime → Change runtime type → T4 GPU → Run all.**

> If you don't have a T4 GPU, the script will detect this and fall back to
> `scripts/run_training_demo.py`, which uses a heuristic policy with
> epsilon decay and produces real (noisy) curves against the live env.

In [ ]:
# 1. Clone the repo and cd into it (so `from server.app import app` works)
import os
if not os.path.isdir("netweaver-sre"):
    !git clone https://github.com/Shasidharyadav/netweaver-sre.git
%cd netweaver-sre

# 2. Install dependencies
!pip install -q openenv-core trl matplotlib datasets transformers huggingface_hub torch requests fastapi uvicorn pydantic httpx pillow

In [ ]:
import os
import sys
import time
import threading
import uvicorn
import requests

# Ensure repo root is importable
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.environ.setdefault("ENV_URL", "http://0.0.0.0:8000")

# Boot the FastAPI server in a daemon thread
from server.app import app  # noqa: E402

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Wait until the /health endpoint is alive
for _ in range(30):
    try:
        r = requests.get("http://0.0.0.0:8000/health", timeout=1)
        if r.status_code == 200:
            print("Server ready:", r.json())
            break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("Server failed to start within 15s")

In [ ]:
import os
import json

# Optional: set your HF token (for gated/larger models). Otherwise leave empty.
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Speed knobs (lower = faster). Override in env to scale up.
os.environ.setdefault("MAX_TRAIN_STEPS", "50")
os.environ.setdefault("EVAL_EPISODES", "10")
os.environ.setdefault("MAX_STEPS_PER_EPISODE", "8")

# Run real GRPO if we have a GPU + transformers/trl, else fall back to the
# heuristic training proxy that produces real noisy curves against the env.
import train_grpo
train_grpo.main()

with open("training_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

print("Avg before:", sum(results["before_rewards"]) / len(results["before_rewards"]))
print("Avg after :", sum(results["after_rewards"])  / len(results["after_rewards"]))
print("Source    :", results.get("source", "unknown"))

In [ ]:
# Inline plots: reward curve + before/after + difficulty breakdown
import json
import matplotlib.pyplot as plt

with open("training_results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

tr = data.get("training_rewards", [])
before = data.get("before_rewards", [])
after = data.get("after_rewards", [])
diff = data.get("difficulty_breakdown", {})

# Reward curve with moving average
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(tr) + 1), tr, alpha=0.55, color="#2563eb", label="Per-episode reward")
if len(tr) >= 5:
    ma = [sum(tr[max(0, i - 4):i + 1]) / min(5, i + 1) for i in range(len(tr))]
    plt.plot(range(1, len(ma) + 1), ma, color="#dc2626", linewidth=2.5, label="5-step moving avg")
    plt.legend()
plt.title("Training Reward Curve")
plt.xlabel("Step")
plt.ylabel("Reward (rubric score)")
plt.grid(alpha=0.3)
plt.show()

# Before vs after
plt.figure(figsize=(5, 4))
b = sum(before) / len(before) if before else 0.001
a = sum(after) / len(after) if after else 0.001
plt.bar(["Before", "After"], [b, a], color=["#7c3aed", "#16a34a"])
plt.ylim(0, 1)
plt.title("Before vs After Training")
plt.ylabel("Average Reward")
for i, v in enumerate([b, a]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.show()

# Per-difficulty breakdown
plt.figure(figsize=(7, 4))
labels = ["easy", "medium", "hard"]
bvals = [diff.get("before", {}).get(k, 0.001) for k in labels]
avals = [diff.get("after", {}).get(k, 0.001) for k in labels]
x = list(range(len(labels)))
w = 0.36
plt.bar([i - w / 2 for i in x], bvals, width=w, color="#9333ea", label="Before")
plt.bar([i + w / 2 for i in x], avals, width=w, color="#16a34a", label="After")
plt.xticks(x, [s.title() for s in labels])
plt.ylim(0, 1)
plt.title("Reward by Difficulty")
plt.ylabel("Average Reward")
plt.legend()
plt.show()